# Module 12 — RAG + chat (retrieve hybride → Ollama)

Passer de **deux moteurs de recherche** (BM25 + vecteurs, module 10–11) à une **réponse conversationnelle sourcée** sur ton corpus.

**Prérequis** :
- `docker compose up -d` (OpenSearch, Qdrant)
- Corpus indexé : `uv run presslake index` + `uv run presslake embed`
- Ollama sur l'hôte : `systemctl status ollama`, `ollama pull llama3.2:1b`
- CPU recommandé sur iGPU AMD : `sudo ./scripts/setup-ollama-cpu-only.sh`

Tuto : [`docs/modules/12-rag-chat.md`](../docs/modules/12-rag-chat.md)

## Étape 0 — Le pipeline en une image

```
Question utilisateur
       │
       ▼
┌──────────────────┐     BM25 (mots exacts)     OpenSearch
│ retrieve_passages│ ◄──────────────────────────┐
│   fusion RRF     │     Vecteurs (paraphrase)  Qdrant + embed
└────────┬─────────┘
         │ passages [1]…[k]
         ▼
┌──────────────────┐
│  build_prompt    │  system + extraits numérotés + question
└────────┬─────────┘
         ▼
┌──────────────────┐
│     Ollama       │  génération locale (CPU/GPU)
└────────┬─────────┘
         ▼
   Réponse + footer sources
```

**Pourquoi pas Ollama seul ?** Sans retrieve, le LLM n'a pas ton corpus — il hallucine. PressLake injecte les extraits **avant** l'appel LLM.

## Étape 1 — Santé des services

Même logique que les modules 10–11 : vérifier les briques avant le RAG.

In [ ]:
import httpx

from presslake.rag.config import ollama_base_url, ollama_model, rag_top_k
from presslake.rag.ollama import check_ollama_available
from presslake.search.client import get_opensearch_client
from presslake.search.index import count_documents
from presslake.vector.client import get_qdrant_client
from presslake.vector.collection import COLLECTION_CHUNKS, count_points

os_client = get_opensearch_client()
qd_client = get_qdrant_client()

print("OpenSearch documents :", count_documents(os_client))
print("Qdrant points        :", count_points(qd_client), f"({COLLECTION_CHUNKS})")
print("Ollama URL           :", ollama_base_url())
print("Ollama modèle        :", ollama_model())
print("RAG_TOP_K            :", rag_top_k())
print("Ollama répond        :", check_ollama_available())

Si `documents` ou `points` = 0 → relancer `presslake index` puis `presslake embed`.

Si Ollama = `False` → `systemctl start ollama` (pas besoin de `ollama serve` si service systemd actif).

## Étape 2 — Helper retrieve

Affiche les passages **avant** le LLM — c'est la brique la plus importante du RAG.

In [ ]:
from presslake.retrieve.hybrid import retrieve_passages


def show_retrieve(
    query: str,
    *,
    limit: int | None = None,
    lang: str | None = None,
    bm25_only: bool = False,
    vector_only: bool = False,
    note: str = "",
) -> list:
    passages = retrieve_passages(
        query,
        limit=limit or rag_top_k(),
        lang=lang,
        bm25_only=bm25_only,
        vector_only=vector_only,
    )
    mode = "hybride RRF"
    if bm25_only:
        mode = "BM25 seul"
    elif vector_only:
        mode = "vecteur seul"
    lang_hint = f" lang={lang}" if lang else ""
    print(f"Requête : {query!r} | mode={mode}{lang_hint}")
    if note:
        print(f"Note    : {note}")
    print(f"Passages: {len(passages)}\n")
    for i, p in enumerate(passages, 1):
        src = "+".join(p.sources)
        chunk = f"chunk {p.chunk_index}" if p.chunk_index is not None else "article"
        print(f"{i}. [RRF {p.score:.4f}] [{src}] {p.feed_id} | {chunk} | {p.citation_label()}")
        print(f"   … {(p.text or '')[:120]}…")
        if p.canonical_url:
            print(f"   url: {p.canonical_url}")
    print()
    return passages

### Cas A — Retrieve hybride (question corpus)

**Attendu** : passages sur le Népal avec sources `bm25`, `vector` ou `bm25+vector`.

In [ ]:
passages_nepal = show_retrieve(
    "Que s'est-il passé au Népal ? inondations catastrophe",
    note="Hybride — fusion RRF des deux moteurs",
)

### Cas B — BM25 seul vs vecteur seul vs hybride

Compare les **trois modes** sur la même requête paraphrasée (cas E du module 10 / A du module 11).

In [ ]:
query_para = "catastrophe himalayenne crue meurtrière"

print("=== BM25 seul ===")
show_retrieve(query_para, bm25_only=True, limit=5)

print("=== Vecteur seul ===")
show_retrieve(query_para, vector_only=True, limit=5)

print("=== Hybride RRF ===")
show_retrieve(query_para, limit=5, note="RRF remonte ce que chaque moteur fait bien")

**À retenir** : BM25 = mots exacts ; vecteur = sens proche ; RRF **fusionne les classements** sans normaliser les scores bruts.

### Cas C — Inspecter le prompt (sans LLM)

Avant d'appeler Ollama, regarde ce que le modèle reçoit vraiment.

In [ ]:
from presslake.rag.prompt import SYSTEM_PROMPT, build_chat_messages

question = "Que dit le corpus sur le Népal ?"
messages = build_chat_messages(question, passages_nepal[:4])

print("=== SYSTEM (extrait) ===")
print(SYSTEM_PROMPT[:200], "…\n")
print("=== USER (extrait) ===")
user_msg = messages[1]["content"]
print(user_msg[:1200])
if len(user_msg) > 1200:
    print(f"\n… [{len(user_msg) - 1200} caractères tronqués] — prompt complet = {len(user_msg)} car.")

Plus `RAG_TOP_K` est grand, plus le prompt est long → plus lent (surtout en CPU). `.env` : `RAG_TOP_K=4` par défaut optimisé.

### Cas D — Retrieve seul via `answer_question` (sans Ollama)

Équivalent CLI : `presslake chat "…" --retrieve-only`

In [ ]:
from presslake.rag.chat import answer_question

preview = answer_question(question, skip_llm=True)
print(preview.answer)

### Cas E — RAG complet (Ollama)

**Attendu** : réponse avec citations `[1]`, `[2]` + footer **Sources PressLake**.

⏱ Première exécution : chargement embed (~10 s) + retrieve + génération CPU (~15–30 s).

In [ ]:
if not check_ollama_available():
    print("Ollama indisponible — sauter ce cas ou lancer : systemctl start ollama")
else:
    result = answer_question("Que dit le corpus sur le Népal ?")
    print("Refusé :", result.refused)
    print("Modèle :", result.model)
    print("Passages :", len(result.passages))
    print("\n--- Réponse ---\n")
    print(result.answer)

### Cas F — Refus hors corpus

**Attendu** : pas d'appel LLM utile — message de refus explicite (retrieve vide).

In [ ]:
off_topic = answer_question("Quel est le cours du bitcoin hier soir ?")
print("Refusé :", off_topic.refused)
print(off_topic.answer)

## Étape 3 — API PressLake (optionnel)

Si `uv run presslake serve --host 0.0.0.0` tourne dans un autre terminal :

In [ ]:
API = "http://localhost:8000"

try:
    health = httpx.get(f"{API}/health", timeout=3.0)
    health.raise_for_status()
    print("API OK :", health.json())

    models = httpx.get(f"{API}/v1/models", timeout=3.0).json()
    print("Modèles /v1 :", [m["id"] for m in models.get("data", [])])

    chat = httpx.post(
        f"{API}/chat",
        json={"message": "Résumé en une phrase : actualité Népal dans le corpus"},
        timeout=180.0,
    )
    chat.raise_for_status()
    data = chat.json()
    print("\nRéponse API (extrait) :", data.get("answer", "")[:400], "…")
except httpx.HTTPError as exc:
    print(f"API indisponible ({exc}) — lancer : uv run presslake serve --host 0.0.0.0")

## Étape 4 — Open WebUI (hors notebook)

1. `docker compose up -d open-webui` → http://localhost:3000
2. Modèle **`presslake-rag`** (pointe vers PressLake `/v1`, pas Ollama direct)
3. Même question que le cas E — tu dois voir les mêmes sources

Si écran noir / crash : `sudo ./scripts/setup-ollama-cpu-only.sh` puis redémarrer `serve`.

## Synthèse — ce que tu dois retenir

| Étape | Rôle | Commande / code |
|---|---|---|
| Retrieve | Trouver les extraits citables | `retrieve_passages()` / `presslake retrieve` |
| Prompt | Injecter le corpus dans le LLM | `build_chat_messages()` |
| LLM | Synthèse en langage naturel | Ollama via `answer_question()` |
| Refus | Pas de hallucination hors index | retrieve vide → pas de génération |
| UI | Client interchangeable | Open WebUI → `/v1/chat/completions` |

**Prochaine étape (module 13)** : Langfuse — tracer les requêtes RAG et mesurer la qualité (hallucinations, latence).